# Lendo os dados do fornecedor com um contrato

## Task 2: ler os três arquivos do fornecedor com um schema declarado no projeto

## Os arquivos do fornecedor

O fornecedor de dados da Arena Analytics inicialmente mandou três arquivos CSV:

| Arquivo | O que tem | Uma linha é... |
|---|---|---|
| `WorldCups.csv` | as edições da Copa: sede, campeão, vice, total de gols | uma Copa |
| `WorldCupMatches.csv` | as partidas: fase, seleções, placar, público, data | uma partida |
| `WorldCupPlayers.csv` | as escalações: seleção, jogador, posição | um jogador numa partida |

Esses arquivos são a porta de entrada do projeto e voltam a chegar a cada Copa nova. Por isso a leitura precisa de
um **contrato**: o projeto declara as colunas e os tipos que espera de cada arquivo.

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-leitura-fornecedor")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

spark

## 1. Leitura sem opções

In [2]:
RAW = "../../data/raw"

data = spark.read.csv(f"{RAW}/WorldCups.csv")

data.printSchema()
data.show()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)

+----+-------------------+------------+--------------+-------------+------------+-----------+--------------+-------------+----------+
| _c0|                _c1|         _c2|           _c3|          _c4|         _c5|        _c6|           _c7|          _c8|       _c9|
+----+-------------------+------------+--------------+-------------+------------+-----------+--------------+-------------+----------+
|Year|            Country|      Winner|    Runners-Up|        Third|      Fourth|GoalsScored|QualifiedTeams|MatchesPlayed|Attendance|
|1930|            Uruguay|     Uruguay|     Argentina|United States|  Yugoslavia|         70|            13|

Repare nos nomes das colunas e nos tipos. `printSchema()` mostra o **schema**: as colunas, o tipo de cada uma e se aceitam nulo (`nullable`).

## 2. Leitura com cabeçalho e inferência de tipos

In [3]:
data = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{RAW}/WorldCups.csv")
)

data.printSchema()

root
 |-- Year: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- Winner: string (nullable = true)
 |-- Runners-Up: string (nullable = true)
 |-- Third: string (nullable = true)
 |-- Fourth: string (nullable = true)
 |-- GoalsScored: integer (nullable = true)
 |-- QualifiedTeams: integer (nullable = true)
 |-- MatchesPlayed: integer (nullable = true)
 |-- Attendance: string (nullable = true)



Compare os nomes e os tipos com a leitura anterior. A inferência ajuda, mas tem três riscos:

1. **Custo:** para descobrir os tipos, o Spark lê o arquivo inteiro uma vez a mais.
2. **Pode errar:** uma linha diferente basta para uma coluna inteira virar texto. Confira o tipo de `Attendance`
   (público).
3. **Não é contrato:** se o fornecedor mudar o formato, a inferência se adapta em silêncio.

## 3. Leitura com schema explícito

O projeto **declara** o schema. No Spark, ele é um `StructType` (a tabela) com uma lista de `StructField` (as
colunas). Cada `StructField` tem o **nome**, o **tipo** e se **aceita nulo**.

In [4]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

schema_copas = StructType([
    StructField("Year",             IntegerType(),      True),
    StructField("Country",          StringType(),       True),
    StructField("Winner",           StringType(),       True),
    StructField("Runners-Up",       StringType(),       True),
    StructField("Third",            StringType(),       True),
    StructField("Fourth",           StringType(),       True),
    StructField("GoalsScored",      IntegerType(),      True),
    StructField("QualifiedTeams",   IntegerType(),      True),
    StructField("MatchesPlayed",    IntegerType(),      True),
    StructField("Attendance",       StringType(),       True)
])

copas = (
    spark.read
    .option("header", "true")
    .schema(schema_copas)
    .csv(f"{RAW}/WorldCups.csv")
)

copas.printSchema()
copas.show(5)

root
 |-- Year: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- Winner: string (nullable = true)
 |-- Runners-Up: string (nullable = true)
 |-- Third: string (nullable = true)
 |-- Fourth: string (nullable = true)
 |-- GoalsScored: integer (nullable = true)
 |-- QualifiedTeams: integer (nullable = true)
 |-- MatchesPlayed: integer (nullable = true)
 |-- Attendance: string (nullable = true)

+----+-----------+------------+--------------+-------------+----------+-----------+--------------+-------------+----------+
|Year|    Country|      Winner|    Runners-Up|        Third|    Fourth|GoalsScored|QualifiedTeams|MatchesPlayed|Attendance|
+----+-----------+------------+--------------+-------------+----------+-----------+--------------+-------------+----------+
|1930|    Uruguay|     Uruguay|     Argentina|United States|Yugoslavia|         70|            13|           18|   590,549|
|1934|      Italy|       Italy|Czechoslovakia|      Germany|   Austria|         70|     

## O custo da inferência, na Spark UI

Cada trabalho que o Spark executa vira um **job**. O `statusTracker()` da sessão conta os jobs: quantos cada forma de
leitura dispara?

In [5]:
rastreador = spark.sparkContext.statusTracker()

antes = len(rastreador.getJobIdsForGroup())
spark.read.option("header", "true").schema(schema_copas).csv(f"{RAW}/WorldCups.csv")
print("jobs com schema explícito:", len(rastreador.getJobIdsForGroup()) - antes)

antes = len(rastreador.getJobIdsForGroup())
spark.read.option("header", "true").option("inferSchema", "true").csv(f"{RAW}/WorldCups.csv")
print("jobs com inferSchema     :", len(rastreador.getJobIdsForGroup()) - antes)

jobs com schema explícito: 0
jobs com inferSchema     : 2


Com o schema, o Spark só anota como ler; a leitura de verdade acontece quando um resultado é pedido (tema da aula 2).
Com o `inferSchema`, ele precisa abrir o arquivo para descobrir os tipos.

Confira na Spark UI (aba **Jobs**). O nome dos jobs
(`csv at DirectMethodHandleAccessor.java:...`) vem do Java, que roda por baixo do PySpark; o que importa é o começo,
`csv`: uma leitura de CSV.

## Os schemas do projeto

Os schemas dos três arquivos ficam em `src/wc_pipeline/schemas.py`: o contrato com o fornecedor, num lugar só do
projeto.

In [6]:
import sys
sys.path.insert(0, "../../src")

from wc_pipeline.schemas import MATCHES, PLAYERS

partidas = spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
escalacoes = spark.read.option("header", "true").schema(PLAYERS).csv(f"{RAW}/WorldCupPlayers.csv")

print(partidas.count())
print(escalacoes.count())

partidas.select("Year", "Stage", "Home Team Name", "Home Team Goals",
                "Away Team Goals", "Away Team Name").show(5)

977
29183
+----+-------+--------------+---------------+---------------+--------------+
|Year|  Stage|Home Team Name|Home Team Goals|Away Team Goals|Away Team Name|
+----+-------+--------------+---------------+---------------+--------------+
|1930|Group 1|        France|              4|              1|        Mexico|
|1930|Group 4| United States|              3|              0|       Belgium|
|1930|Group 2|    Yugoslavia|              2|              1|        Brazil|
|1930|Group 3|       Romania|              3|              1|          Peru|
|1930|Group 1|     Argentina|              1|              0|        France|
+----+-------+--------------+---------------+---------------+--------------+
only showing top 5 rows


`sys.path.insert` indica ao Python onde está o pacote `wc_pipeline`: na pasta `src` do projeto.

## Demanda da aula 1 atendida

| Etapa | Entrega |
|---|---|
| Task 1. Subir o ambiente | o container com Spark 4.2, igual em qualquer máquina |
| Conhecendo o Spark | o que é, por que aguenta o crescimento e como roda no modo local |
| Conhecendo o Spark por dentro | driver e executors, o modo local, a SparkSession e a Spark UI |
| Task 2. Ler os dados do fornecedor | os três arquivos lidos com o schema do projeto |

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [7]:
spark.stop()